In [1]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel

import requests

url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
response = requests.get(url)
text = response.text
print(f"Loading completed ! Corpus length : {len(text)} ")

Loading completed ! Corpus length : 1115394 


In [2]:
tokenizer = Tokenizer(BPE())
tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False)


trainer = BpeTrainer(
    vocab_size=1500, 
    special_tokens=[]
)

tokenizer.train_from_iterator([text], trainer=trainer)

tokenizer.save("weights/shakespeare_bpe_1500.json")
print("BPE training completed. Vocab size is :", tokenizer.get_vocab_size())

#This only has to be done once




BPE training completed. Vocab size is : 1500


In [74]:
import numpy as np
from src.ai_lib import layers
from src.ai_lib import Model, metrics, SequenceDataLoader
from src.ai_lib import optimizers
from src.ai_lib import losses
from src.ai_lib import preprocessing


tokenizer = preprocessing.CustomBPETokenizer(path= "weights/shakespeare_bpe_1500.json")
data_tensor = np.array(tokenizer.encode(text), dtype=np.int32)



In [4]:
train_test_split = 0.9

n = int(train_test_split * len(data_tensor))
train_data = data_tensor[:n]
val_data = data_tensor[n:]

batch_size = 32
block_size = 64  # Numbers of tokens to look back at
steps_per_epoch = 50 

# Creating dataloaders
train_loader = SequenceDataLoader(
    data=train_data, 
    block_size=block_size, 
    batch_size=batch_size, 
    steps_per_epoch=steps_per_epoch
)

val_loader = SequenceDataLoader(
    data=val_data, 
    block_size=block_size, 
    batch_size=batch_size, 
    steps_per_epoch=10 # Less steps for validation
)

In [5]:
d_model = 128
n_heads = 4
d_ff = 512

dropout_rate = 0.1

vocab_size = tokenizer.vocab_size

llm_network = layers.Sequential([
    layers.Embedding(vocab_size=vocab_size, d_model=d_model),
    layers.SineCosEncoder(d_model=d_model, max_len=1000),


    layers.TransformerBlock(n_heads=n_heads, d_model=d_model, d_ff=d_ff, context_window=block_size,
                            is_causal=True, dropout_rate=dropout_rate, RoPE=True),
    layers.TransformerBlock(n_heads=n_heads, d_model=d_model, d_ff=d_ff, context_window=block_size,
                            is_causal=True, dropout_rate=dropout_rate, RoPE=True),
    layers.TransformerBlock(n_heads=n_heads, d_model=d_model, d_ff=d_ff, context_window=block_size,
                            is_causal=True, dropout_rate=dropout_rate, RoPE=True),

    layers.LayerNormalization(n_features=d_model),

    layers.Linear(d_model, n_out=vocab_size)
])

model = Model(llm_network)

In [ ]:
#model.load_weights('weights/microgpt_v2.npz')

Weights loaded : weights/microgpt_v2.npz


In [7]:
learning_rate = 1e-3
weight_decay = 1e-2

optimizer = optimizers.Adam(learning_rate=learning_rate, weight_decay=weight_decay)
loss = losses.SoftmaxCrossEntropy()

In [64]:
optimizer.learning_rate = 3e-4
# In order to modify the learning rate in between multiple runs of the model.fit cell,
# only change it that way, do not create a new instance of the optimizer as it would delete momentum etc

In [65]:
print("Beginning of training of microgpt_v2")
model.fit(
    dataloader=train_loader,
    epochs=20,
    loss=loss,
    optimizer=optimizer,
    validation_dataloader=val_loader,
    verbose=True,
    metrics=[metrics.accuracy]
)

Beginning of training of microgpt_v2

--- Epoch : 0 ---
Training loss : 3.34559
  accuracy (train) : 0.30592
Validation loss : 4.00551
  accuracy (val) : 0.26826

--- Epoch : 1 ---
Training loss : 3.35722
  accuracy (train) : 0.30285
Validation loss : 3.98484
  accuracy (val) : 0.26411

--- Epoch : 2 ---
Training loss : 3.33021
  accuracy (train) : 0.30850
Validation loss : 4.02467
  accuracy (val) : 0.26792

--- Epoch : 3 ---
Training loss : 3.33353
  accuracy (train) : 0.30815
Validation loss : 4.01828
  accuracy (val) : 0.27090

--- Epoch : 4 ---
Training loss : 3.33123
  accuracy (train) : 0.30828
Validation loss : 4.03918
  accuracy (val) : 0.26763

--- Epoch : 5 ---
Training loss : 3.31169
  accuracy (train) : 0.31057
Validation loss : 4.00266
  accuracy (val) : 0.27100

--- Epoch : 6 ---
Training loss : 3.31977
  accuracy (train) : 0.30876
Validation loss : 3.95748
  accuracy (val) : 0.27202

--- Epoch : 7 ---
Training loss : 3.33155
  accuracy (train) : 0.30812
Validation loss 

In [72]:
def generate_text(model, tokenizer, prompt_text="A ", max_new_tokens=100):
    """
    Generates text
    """
    # Important : Turns off setting
    model.sequential.set_training(False)
    
    # Use of kv_cache for inference
    model.sequential.set_use_cache(True)
    model.sequential.reset_cache()
    
    # Encodes context (initial text / prompt)
    context_ids = tokenizer.encode(prompt_text)
    
    # Prefill
    X_input = np.array([context_ids], dtype=np.int32)
    logits = model.predict(X_input)
    
    # We take the predicted char and add it to the list
    next_char_id = sample_top_k(logits[0, -1, :], temperature=1.0, top_k=3)
    context_ids.append(next_char_id)
    
    # Decode
    for _ in range(max_new_tokens - 1):
        # X_input becomes a simple integer as we use kv_cache
        X_input = np.array([[next_char_id]], dtype=np.int32)
        
        logits = model.predict(X_input)
        
        # Since X_input is of length one, logits has shape (1, 1, Vocab_size)
        next_char_id = sample_top_k(logits[0, -1, :], temperature=1.0, top_k=3)
        context_ids.append(next_char_id)
        
    return tokenizer.decode(context_ids)


def sample_top_k(logits, temperature=1.5, top_k=5):
    """
    Sampling from only the k most likely tokens with temperature
    """

    logits = logits / temperature
    
    indices_to_remove = np.argsort(logits)[:-top_k]
    
    logits[indices_to_remove] = -np.inf
    
    exp_preds = np.exp(logits - np.max(logits))
    preds = exp_preds / np.sum(exp_preds)
    
    return np.random.choice(len(logits), p=preds)

generated_text = generate_text(model, tokenizer, prompt_text="KING", max_new_tokens=80)
print(generated_text)

KING HENRY VI:
I will tell thee, my lord; for I will not be,
And I will not die to me, but I am not
After'd to the Duke of York, and the Duke of York

KING EDWARD IV:
Say, that thou art dead?

KING RICHARD III IV:
Vicizes,
E'
Cl


In [ ]:
# It seems that leaving attention sinks and using RoPE to allow for a sliding window didn't allow the model to generalize

In [73]:
model.save_weights("weights/microgpt_v2.npz")

Model saved : weights/microgpt_v2.npz
